<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/FrozenLake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Aprendizaje por Refuerzo: Q-Learning con Frozen Lake**
## **Cómo aprende un agente experimentando**

## Índice

### 1. Introducción: Aprender por Ensayo y Error

* ¿Qué es el Reinforcement Learning (RL)?
* Diferencias con el Aprendizaje Supervisado (aprender de un jefe vs. aprender de la experiencia).
* El ciclo Agente ↔ Entorno ↔ Recompensa.

### 2. El Escenario: Frozen Lake y Gymnasium

* Presentación del problema: Hielo, agujeros y una meta.
* El concepto de **Entorno Estocástico**: ¿Por qué a veces el agente resbala?
* **💻 Código:** Instalación de `gymnasium` y creación del entorno.

### 3. Conceptos Clave: El Vocabulario del RL

* Estado, Acción y Recompensa.
* La Política (π): El "manual de instrucciones" que el agente debe escribir.
* El objetivo: Maximizar la recompensa acumulada.

### 4. La Q-Table: La Memoria del Agente

* ¿Qué es realmente una Q-Table? Una "hoja de trucos" de 16×4.
* **💻 Código:** Inicialización de la tabla con `numpy.zeros`.

### 5. La Ecuación de Bellman: ¿Cómo aprende el cerebro?

* Explicación intuitiva: "Actualizo lo que creía saber con lo que acabo de descubrir".
* La fórmula en LaTeX.
* Explicación de los "mandos de control": α (alfa) ritmo de aprendizaje y γ (gamma) visión a futuro.

### 6. Explorar o Explotar: El Dilema -greedy

* El dilema del agente: ¿Hago lo que ya sé que funciona o pruebo algo nuevo?
* Implementación del decaimiento de epsilon (ε-decay).

### 7. ¡A Entrenar! Enseñando al Agente a Sobrevivir

* **💻 Código:** El bucle de entrenamiento completo.
* Explicación de los pasos: `reset`, `step`, `update`, `render`.

### 8. Análisis de Resultados: ¿Ha aprendido nuestro agente?

* **💻 Código:** Gráfica de la curva de aprendizaje (recompensas por episodio).
* **💻 Código:** Visualización pro mediante un **Heatmap** de la Q-Table (¿Qué zonas considera peligrosas?).
* **💻 Código:** Demo: Ver al agente jugar.

### 9. Conclusión y Siguientes Pasos

* Limitaciones de las tablas (¿Qué pasa si el mundo es gigante?).
* Introducción al Deep Q-Learning (DQN) para triunfar en los juegos de Atari.

---


# **1. Introducción: Aprender por Ensayo y Error**

¡Bienvenido al fascinante mundo del **Reinforcement Learning (RL)** o Aprendizaje por Refuerzo! Si alguna vez has entrenado a un perro para que se siente a cambio de una golosina, o si has aprendido a montar en bicicleta cayéndote una cuantas veces antes de mantener el equilibrio, ya entiendes los principios básicos de esta rama de la Inteligencia Artificial.

### ¿Qué es el Reinforcement Learning?

A diferencia de otras áreas de la IA, el RL no se trata de predecir una etiqueta (como decir si una foto es un gato o un perro) ni de encontrar patrones ocultos en los datos. El RL trata sobre la **toma de decisiones secuenciales**.

El **Reinforcement Learning** es una rama del Machine Learning donde un **Agente** aprende a tomar decisiones interactuando con un **Entorno**. A diferencia de otros métodos de aprendizaje automático, el Agente:

- ❌ **No tiene un profesor que le diga exactamente qué hacer en cada situación**
- ✅ **Aprende de las consecuencias de sus propias acciones**
- ✅ **Recibe señales de "recompensa" o "castigo" que le indican si lo está haciendo bien o mal**

Es como aprender a jugar al ajedrez: nadie te dice "en esta posición exacta, mueve el caballo aquí". En cambio, juegas muchas partidas, pierdes, ganas, y poco a poco descubres qué estrategias funcionan mejor.

---

### Aprendizaje Supervisado vs. RL: El Jefe vs. La Experiencia

Para entenderlo mejor, comparemos el RL con el aprendizaje más tradicional:

* **Aprendizaje Supervisado (El "Jefe"):** Imagina que tienes un jefe que te da una lista de tareas y te dice exactamente cómo debe quedar cada una. Si te equivocas, te corrige al instante. Tienes ejemplos claros de "entrada" y "salida".
* **Reinforcement Learning (La "Experiencia"):** Aquí no hay jefe. Estás solo en una habitación con un objetivo. Pruebas cosas: si algo funciona, recibes una moneda; si algo sale mal, no recibes nada o recibes un pequeño "toque". Nadie te dice *qué* hacer, solo te dicen *qué tan bien* lo hiciste después de que lo intentaste.

> **En resumen:** En el aprendizaje supervisado aprendes de un "maestro". En el RL, aprendes de tu propia **experiencia**.


| **Aprendizaje Supervisado** | **Aprendizaje por Refuerzo** |
|------------------------------|------------------------------|
| **"Aprender de un profesor"** | **"Aprender de la experiencia"** |
| Tienes datos etiquetados: "esta imagen es un gato" | No hay etiquetas, solo señales de recompensa |
| El objetivo es **imitar** ejemplos correctos | El objetivo es **descubrir** la mejor estrategia |
| Aprendes de datos estáticos | Aprendes interactuando con un entorno dinámico |
| Ejemplo: Clasificar imágenes de perros y gatos | Ejemplo: Enseñar a un robot a caminar |

---


### El ciclo fundamental del RL

Un sistema de Reinforcement Learning logra que el modelo aprenda iterando este ciclo.

```
    ┌─────────────────────────────────────────┐
    │                                         │
    │   👤 AGENTE                             │
    │   (Quien toma las decisiones)           │
    │                                         │
    └──────────┬──────────────────────────────┘
               │                    ▲
               │ Acción             │ Estado +
               │                    │ Recompensa
               ▼                    │
    ┌───────────────────────────────┴─────────┐
    │                                         │
    │   🌍 ENTORNO                            │
    │   (El mundo donde actúa el agente)      │
    │                                         │
    └─────────────────────────────────────────┘
```


#### Los cinco componentes fundamentales:

1. **🤖 El AGENTE** (Agent)
   - Es quien aprende y toma decisiones
   - En nuestro caso: el algoritmo de Q-Learning
   - Piensa en él como "el jugador"

2. **🌍 El ENTORNO** (Environment)
   - Es el mundo donde actúa el agente
   - Define las reglas del juego
   - En nuestro caso: el lago congelado de Frozen Lake
   - Puede ser determinista (siempre pasa lo mismo) o estocástico (tiene aleatoriedad)

3. **📍 El ESTADO** (State - *s*)
   - Es la situación actual en la que se encuentra el agente
   - Representa "dónde estoy" o "qué está pasando ahora"
   - En Frozen Lake: La casilla específica donde está el agente
   - Ejemplo: _"Estoy en la casilla (2,3)"_

4. **🎯 La ACCIÓN** (Action - *a*)
   - Es la decisión que toma el agente
   - Representa "qué voy a hacer"
   - En Frozen Lake: Moverse en una de cuatro direcciones (←, ↓, →, ↑)
   - Ejemplo: _"Voy a moverme hacia arriba"_

5. **🎁 La RECOMPENSA** (Reward - *r*)
   - Es la señal de feedback que recibe el agente después de cada acción
   - Le dice "esto estuvo bien" (+) o "esto estuvo mal" (-)
   - En Frozen Lake: +1 por llegar a la meta, 0 en cualquier otro caso
   - Es el "profesor silencioso" que guía el aprendizaje

<img src="https://github.com/financieras/math_for_ai/blob/main/img/reinforcement_learning.jpeg?raw=1" alt="reinforcement learning" width="480"/>

---

#### ¿Cómo funciona el ciclo completo?

Ahora que conocemos los componentes, veamos cómo interactúan en cada paso:

1. **📍 El Agente observa el ESTADO actual** del entorno
   - _"Estoy en la casilla (0,0) del lago"_

2. **🎯 El Agente elige una ACCIÓN** basándose en su conocimiento actual y se ejecuta la acción.
   - _"Voy a moverme hacia la derecha"_

3. **🌍 El Entorno responde**:
   - Cambia el Estado
   - Hay un nuevo **ESTADO**: _"Ahora estás en la casilla (0,1)"_
   - Le da una **RECOMPENSA**  (puede ser positiva, negativa o cero):
   - _"0 puntos (aún no llegas a la meta)"_

4. **🤖 El Agente aprende** de esta experiencia
   - Actualiza su conocimiento: _"Ah, moverme a la derecha desde (0,0) me llevó a (0,1) sin obtener recompensa"_

5. **🔄 Se repite el ciclo** desde el paso 1 con el nuevo estado
   - El agente continúa hasta alcanzar un **estado terminal**: llegar a la meta 🎯 o caer en un agujero ❌

---

Este ciclo completo (desde el inicio hasta un estado terminal) se llama un **episodio**. El agente jugará **miles de episodios**, y con cada uno se volverá más inteligente al descubrir qué acciones llevan a mejores recompensas. Así el agente descubre una buena estrategia (lo que llamamos política).

**Piénsalo así:**
- Un episodio = Una partida completa del juego
- Miles de episodios = El entrenamiento completo del agente

---

### ¿Listo para empezar?

Ahora que comprendes la filosofía del Reinforcement Learning, es momento de ponernos manos a la obra. En la siguiente sección, conocerás el **entorno Frozen Lake**, donde nuestro agente aprenderá a sobrevivir en un lago congelado lleno de peligros.

🧊 ¡Vamos a patinar sobre hielo! 🧊

---

**Frozen Lake**.

Un agente deberá cruzar un lago congelado evitando agujeros… ¡sin que nadie le diga por dónde ir!

Solo recibirá una recompensa +1 cuando llegue a la meta… y -1 si cae al agua.  
Todo lo demás lo tendrá que aprender **por ensayo y error**.